# Process to be used by frontend code

Initial quick iteration to plot clusters on UI from Google Colab outputs

In [1]:
import json
import pandas as pd
import numpy as np
from sklearn.decomposition import PCA
from collections import defaultdict
from sklearn.metrics.pairwise import euclidean_distances
import ast

In [2]:
def process_cluster_file(filepath, artist_id_col="artist_name"):
    with open(filepath) as f:
        data = json.load(f)
    
    all_artist_rows = []
    all_cluster_positions = []

    for version in data:
        n = version["n"]
        n_clusters = len(version["clusters"])
        dist_cols = [f"dist_to_cluster_{c}" for c in range(n_clusters)]

        # Flatten all points across all clusters
        rows = []
        for cluster in version["clusters"]:
            for point in cluster["data"]:
                row = {"n": n, "cluster": int(cluster["key"]), "keywords": cluster["keywords"]}
                row.update(point)
                rows.append(row)
        
        df = pd.DataFrame(rows)

        # Normalize artist id col — institutions file uses a list in "artists"
        # so we explode it so each artist in a shared exhibition gets their own row
        if "artists" in df.columns and artist_id_col == "artists":
            df = df.explode("artists")
            group_col = "artists"
        else:
            group_col = artist_id_col

        # Keyword map per cluster for easy lookup
        keyword_map = {
            int(c["key"]): c["keywords"] for c in version["clusters"]
        }

        for artist_id, group in df.groupby(group_col):
            total_points = len(group)
            cluster_counts = group["cluster"].value_counts()
            primary_cluster = int(cluster_counts.idxmax())

            # Soft membership: % of points in each cluster
            cluster_distribution = {
                str(c): round(cluster_counts.get(c, 0) / total_points, 4)
                for c in range(n_clusters)
            }

            # Average distance to every cluster center across all points
            avg_dists = {
                f"avg_dist_to_cluster_{c}": round(group[f"dist_to_cluster_{c}"].mean(), 4)
                for c in range(n_clusters)
            }

            # Compute cluster center positions in 2D
            center_vectors = np.array([
                [df[df["cluster"] == c][f"dist_to_cluster_{cc}"].mean() for cc in range(n_clusters)]
                for c in range(n_clusters)
            ])
            reducer = PCA(n_components=2)
            centers_2d = reducer.fit_transform(center_vectors)
            # Normalize to [0, 1]
            centers_2d -= centers_2d.min(axis=0)
            centers_2d /= centers_2d.max(axis=0)

            # Closest cluster by avg distance (may differ from primary by point count)
            closest_by_dist = int(np.argmin([avg_dists[f"avg_dist_to_cluster_{c}"] for c in range(n_clusters)]))

            all_artist_rows.append({
                "n": n,
                "artist": artist_id,
                "total_points": total_points,
                "primary_cluster": primary_cluster,           # cluster with most points
                "closest_cluster_by_dist": closest_by_dist,  # cluster with lowest avg dist
                "cluster_distribution": cluster_distribution, # soft membership
                "cluster_keywords": {                         # keywords for each cluster
                    str(c): keyword_map[c] for c in range(n_clusters)
                },
                **avg_dists
            })
            
        all_cluster_positions.append({
            "n": n,
            "cluster_positions": {
                str(c): {"x": round(float(centers_2d[c][0]), 4), "y": round(float(centers_2d[c][1]), 4)}
                for c in range(n_clusters)
            }
        })

    return pd.DataFrame(all_artist_rows), all_cluster_positions

In [3]:
def add_ring_layout(input_path, output_path):
    with open(input_path) as f:
        data = json.load(f)
    
    # Group rows by n, then by primary_cluster
    by_n = defaultdict(list)
    for row in data:
        by_n[row["n"]].append(row)
    
    for n, rows in by_n.items():
        by_cluster = defaultdict(list)
        for row in rows:
            by_cluster[row["primary_cluster"]].append(row)
        
        for c, cluster_rows in by_cluster.items():
            dist_key = f"avg_dist_to_cluster_{c}"
            sorted_rows = sorted(cluster_rows, key=lambda r: r.get(dist_key, 0.5))
            n_items = len(sorted_rows)
            
            layer_count = min(5, max(2, int(np.ceil(np.sqrt(n_items)))))
            per_layer = int(np.ceil(n_items / layer_count))
            r_min_frac, r_max_frac = 0.16, 1.0
            
            for L in range(layer_count):
                layer_slice = sorted_rows[L * per_layer:(L + 1) * per_layer]
                if not layer_slice:
                    continue
                m = len(layer_slice)
                frac = L / max(layer_count - 1, 1)
                r_frac = r_min_frac + frac * (r_max_frac - r_min_frac)
                phase = L * (np.pi / max(m, 6))
                for j, row in enumerate(layer_slice):
                    row["_angle"] = round(float((j / max(m, 1)) * 2 * np.pi + phase), 4)
                    row["_radius_frac"] = round(float(r_frac), 4)  # multiply by CLUSTER_RADIUS px on frontend
    
    with open(output_path, "w") as f:
        json.dump(data, f, indent=2)
    print(f"Done — {len(data)} rows written to {output_path}")

In [4]:
REFERENCE_W = 1200
REFERENCE_H = 800
CANVAS_PADDING = 220
CLUSTER_RADIUS_BASE = 210
COLLISION_GAP = 40

def resolve_cluster_positions(positions_path, summary_path, output_path):
    with open(positions_path) as f:
        positions_data = json.load(f)
    with open(summary_path) as f:
        summary_data = json.load(f)
    
    counts = defaultdict(lambda: defaultdict(int))
    for row in summary_data:
        counts[row["n"]][row["primary_cluster"]] += 1
    
    resolved = []
    
    for entry in positions_data:
        n = entry["n"]
        cluster_positions = entry["cluster_positions"]
        n_clusters = len(cluster_positions)

        scale = max(1.0, 1 + (n - 6) * 0.18)  # starts growing past n=6
        ref_w = int(REFERENCE_W * scale)
        ref_h = int(REFERENCE_H * scale)
        gap = COLLISION_GAP + max(0, (n - 6) * 12)  # extra gap per cluster above 6
        
        inner_w = ref_w - CANVAS_PADDING * 2
        inner_h = ref_h - CANVAS_PADDING * 2
        
        pos = []
        for c in range(n_clusters):
            p = cluster_positions.get(str(c), {"x": 0.5, "y": 0.5})
            artist_count = counts[n].get(c, 1)
            footprint = min(CLUSTER_RADIUS_BASE, 160 + min(52, artist_count * 5))
            pos.append({
                "c": c,
                "x": CANVAS_PADDING + float(p["x"]) * inner_w,
                "y": CANVAS_PADDING + float(p["y"]) * inner_h,
                "r": footprint,
            })
        
        iterations = 300 + max(0, (n - 6) * 80)
        for _ in range(iterations):
            for i in range(n_clusters):
                for j in range(i + 1, n_clusters):
                    dx = pos[j]["x"] - pos[i]["x"]
                    dy = pos[j]["y"] - pos[i]["y"]
                    d = max(np.hypot(dx, dy), 0.001)
                    need = pos[i]["r"] + pos[j]["r"] + gap
                    if d < need:
                        push = (need - d) / 2
                        pos[i]["x"] -= (dx / d) * push
                        pos[i]["y"] -= (dy / d) * push
                        pos[j]["x"] += (dx / d) * push
                        pos[j]["y"] += (dy / d) * push
        
        all_x = [p["x"] for p in pos]
        all_y = [p["y"] for p in pos]
        margin = 80
        canvas_min_w = int(max(ref_w, max(all_x) + CANVAS_PADDING + margin))
        canvas_min_h = int(max(ref_h, max(all_y) + CANVAS_PADDING + margin))

        resolved_positions = {}
        for p in pos:
            nx = np.clip((p["x"] - CANVAS_PADDING) / inner_w, 0.02, 0.98)
            ny = np.clip((p["y"] - CANVAS_PADDING) / inner_h, 0.02, 0.98)
            resolved_positions[str(p["c"])] = {
                "x": round(float(nx), 4),
                "y": round(float(ny), 4),
            }
        
        resolved.append({
            "n": n,
            "cluster_positions": resolved_positions,
            "canvas_min_w": canvas_min_w, 
            "canvas_min_h": canvas_min_h, 
            "aspect_ratio": round(canvas_min_w / canvas_min_h, 4)
        })
    
    with open(output_path, "w") as f:
        json.dump(resolved, f, indent=2)

In [5]:
institution_artist_df, institution_positions = process_cluster_file("../../data/colab/all_clusters_inst.json", artist_id_col="artists")
artist_artist_df, artist_positions = process_cluster_file("../../data/colab/all_clusters_ar.json", artist_id_col="artist_id")

# export
institution_artist_df.to_json("../../data/ui/institution_artist_summary.json", orient="records", indent=2)
artist_artist_df.to_json("../../data/ui/artist_artist_summary.json", orient="records", indent=2)

with open("../../data/ui/institution_cluster_positions.json", "w") as f:
    json.dump(institution_positions, f, indent=2)
with open("../../data/ui/artist_cluster_positions.json", "w") as f:
    json.dump(artist_positions, f, indent=2)

add_ring_layout("../../data/ui/artist_artist_summary.json", "../../data/ui/artist_artist_summary.json")
add_ring_layout("../../data/ui/institution_artist_summary.json", "../../data/ui/institution_artist_summary.json")

resolve_cluster_positions("../../data/ui/artist_cluster_positions.json", "../../data/ui/artist_artist_summary.json", "../../data/ui/artist_cluster_positions.json")
resolve_cluster_positions("../../data/ui/institution_cluster_positions.json", "../../data/ui/institution_artist_summary.json", "../../data/ui/institution_cluster_positions.json")

print(institution_artist_df.columns.tolist())
print(institution_artist_df.head())

Done — 140 rows written to ../../data/ui/artist_artist_summary.json
Done — 182 rows written to ../../data/ui/institution_artist_summary.json
['n', 'artist', 'total_points', 'primary_cluster', 'closest_cluster_by_dist', 'cluster_distribution', 'cluster_keywords', 'avg_dist_to_cluster_0', 'avg_dist_to_cluster_1', 'avg_dist_to_cluster_2', 'avg_dist_to_cluster_3', 'avg_dist_to_cluster_4', 'avg_dist_to_cluster_5', 'avg_dist_to_cluster_6', 'avg_dist_to_cluster_7', 'avg_dist_to_cluster_8', 'avg_dist_to_cluster_9', 'avg_dist_to_cluster_10', 'avg_dist_to_cluster_11', 'avg_dist_to_cluster_12', 'avg_dist_to_cluster_13', 'avg_dist_to_cluster_14']
   n  artist  total_points  primary_cluster  closest_cluster_by_dist  \
0  2       0            46                1                        1   
1  2       1             7                0                        0   
2  2       2             8                0                        0   
3  2       3             6                1                        1 

In [7]:
df = pd.read_csv("../../data/colab/exhibitions_with_labels.csv")

# Parse artists column — stored as string like "[1, 2, 3]"
df["artist_ids"] = df["artists"].apply(ast.literal_eval)

# Count how many exhibitions each artist appears in
from collections import Counter
all_artist_occurrences = Counter()
for ids in df["artist_ids"]:
    for aid in ids:
        all_artist_occurrences[aid] += 1

# Shared = appears in more than one exhibition → outer ring
# Unique = only in this exhibition → inner ring
R_INNER = 0.35   # _radius_frac for unique artists
R_OUTER = 0.85   # _radius_frac for shared artists

exhibitions_out = []

for _, row in df.iterrows():
    artist_ids = row["artist_ids"]
    labels = ast.literal_eval(row["labels"]) if isinstance(row["labels"], str) else row["labels"]

    # Split into inner/outer
    inner = [aid for aid in artist_ids if all_artist_occurrences[aid] == 1]
    outer = [aid for aid in artist_ids if all_artist_occurrences[aid] > 1]

    items = []

    # Evenly space inner ring
    for j, aid in enumerate(inner):
        angle = (j / max(len(inner), 1)) * 2 * np.pi
        items.append({
            "artistId": aid,
            "_angle": round(float(angle), 4),
            "_radius_frac": R_INNER,
            "isShared": False,
        })

    # Evenly space outer ring, stagger phase slightly from inner
    phase = np.pi / max(len(outer), 6)
    for j, aid in enumerate(outer):
        angle = (j / max(len(outer), 1)) * 2 * np.pi + phase
        items.append({
            "artistId": aid,
            "_angle": round(float(angle), 4),
            "_radius_frac": R_OUTER,
            "isShared": True,
        })

    exhibitions_out.append({
        "id": int(row["id"]),
        "name": row["name"],
        "labels": labels[:3],   # top 2 for display
        "items": items,
    })

with open("../../data/ui/exhibitions_precomputed.json", "w") as f:
    json.dump(exhibitions_out, f, indent=2)

print(f"Done — {len(exhibitions_out)} exhibitions")
for ex in exhibitions_out:
    inner_c = sum(1 for i in ex["items"] if not i["isShared"])
    outer_c = sum(1 for i in ex["items"] if i["isShared"])
    print(f"  {ex['name'][:40]}: {inner_c} inner, {outer_c} outer")

Done — 10 exhibitions
  Part Asian, 100% Hapa: 0 inner, 1 outer
  War Baby / Love Child: Mixed Race Asian : 3 inner, 1 outer
  Ruth Asawa: Citizen of the Universe: 1 inner, 0 outer
  Isamu Noguchi: Archaic/Modern: 0 inner, 1 outer
  Dis/orient: Contemporary Art of the Asia: 1 inner, 0 outer
  Legacies: Asian American Art Movements i: 0 inner, 1 outer
  Oscar yi Hou: East of sun, west of moon: 1 inner, 0 outer
  Biennale Arte 2015: All the World's Futu: 2 inner, 0 outer
  Global Fascisms: 2 inner, 0 outer
  You Sad Legend: 1 inner, 0 outer
